<a href="https://colab.research.google.com/github/jnanasaiparimi/Sentiment-Analysis-using-DistilBERT/blob/main/Fine_tune_a_Sentiment_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
!pip install transformers datasets scikit-learn

In [26]:
from datasets import load_dataset

dataset = load_dataset("amazon_polarity")

In [27]:
small_data = dataset["train"].select(range(800))

In [35]:
data = []
labels = []

for i, item in enumerate(small_data):
    text = item["content"]
    original_label = item["label"]   # 0 or 1

    data.append(text)

    if original_label == 0:
        labels.append(0)   # Negative
    else:
        labels.append(1)   # Positive

In [36]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    data, labels, test_size=0.2, random_state=42
)

In [37]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

In [38]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
test_dataset = Dataset(test_encodings, test_labels)

In [39]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

Step,Training Loss
10,0.705000
20,0.685291
30,0.621801
40,0.488479
50,0.345354
60,0.517480
70,0.481544
80,0.458337
90,0.316064
100,0.140981


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=160, training_loss=0.36817953959107397, metrics={'train_runtime': 1773.9823, 'train_samples_per_second': 0.722, 'train_steps_per_second': 0.09, 'total_flos': 89746662589440.0, 'train_loss': 0.36817953959107397, 'epoch': 2.0})

In [41]:
from sklearn.metrics import accuracy_score, classification_report

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)

print("Accuracy:", accuracy_score(test_labels, preds))
print(classification_report(test_labels, preds))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy: 0.89375
              precision    recall  f1-score   support

           0       0.86      0.92      0.89        73
           1       0.93      0.87      0.90        87

    accuracy                           0.89       160
   macro avg       0.89      0.90      0.89       160
weighted avg       0.90      0.89      0.89       160



In [44]:
texts = [
    "This phone is great",
    "Worst product ever",
    "It's okay, nothing special"
]

inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
outputs = model(**inputs)

probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
predictions = torch.argmax(probs, dim=1)

labels_map = {0:"Negative", 1:"Positive"}

for i in range(len(texts)):
    print(texts[i])
    print("Prediction:", labels_map[predictions[i].item()])
    print("Confidence:", probs[i].max().item())
    print()

This phone is great
Prediction: Positive
Confidence: 0.9836487770080566

Worst product ever
Prediction: Negative
Confidence: 0.9759692549705505

It's okay, nothing special
Prediction: Negative
Confidence: 0.9119232892990112

